In [6]:
import pandas as pd
import numpy as np

# 파일 불러오기
df = pd.read_excel("통합데이터.xlsx")

# -------------------------
# 숫자형 변환
# -------------------------
def to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").strip()
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

for col in ["면적", "고시면적", "토지면적", "출자면적"]:
    df[col] = df[col].apply(to_float)

# -------------------------
# 내부 패턴 판별
# -------------------------
def get_internal_pattern(row):
    g = row["고시면적"]
    t = row["토지면적"]
    c = row["출자면적"]

    g_exists = pd.notna(g)
    t_exists = pd.notna(t)
    c_exists = pd.notna(c)

    if g_exists and t_exists and c_exists:
        if g == t == c:
            return "고시=토지=출자"
        else:
            return None

    if (not g_exists) and t_exists and c_exists:
        if t == c:
            return "고시 없음, 토지=출자"
        else:
            return None

    if g_exists and t_exists and (not c_exists):
        if g == t:
            return "출자 없음, 고시=토지"
        else:
            return None

    if g_exists and (not t_exists) and c_exists:
        if g == c:
            return "토지 없음, 고시=출자"
        else:
            return None

    if g_exists and (not t_exists) and (not c_exists):
        return "고시만"

    if (not g_exists) and t_exists and (not c_exists):
        return "토지만"

    if (not g_exists) and (not t_exists) and c_exists:
        return "출자만"

    return None

df["내부패턴"] = df.apply(get_internal_pattern, axis=1)

# -------------------------
# 내부 대표값
# -------------------------
def get_internal_value(row):
    p = row["내부패턴"]

    if p in ["고시=토지=출자", "출자 없음, 고시=토지", "토지 없음, 고시=출자", "고시만"]:
        return row["고시면적"]

    if p in ["고시 없음, 토지=출자", "토지만"]:
        return row["토지면적"]

    if p == "출자만":
        return row["출자면적"]

    return np.nan

df["내부대표면적"] = df.apply(get_internal_value, axis=1)

# -------------------------
# 최종 체크 조건: 딱 2개만
# -------------------------
def check_target(row):
    reason = str(row.get("토지이동사유", "")).strip()
    eum = row["면적"]          # 토지이음 기준
    internal = row["내부대표면적"]

    # 조건 1: 필지 합병되어 말소
    if "필지 합병되어 말소" in reason:
        return 1

    # 조건 2: 토지이음 면적 != 내부 일치값
    if pd.notna(eum) and pd.notna(internal):
        if float(eum) != float(internal):
            return 1

    return 0

df["체크"] = df.apply(check_target, axis=1)

# -------------------------
# 결과 정리
# -------------------------
result = df[[
    "체크",
    "소재지",
    "면적",
    "토지이동사유",
    "고시면적",
    "토지면적",
    "출자면적",
    "내부패턴",
    "내부대표면적"
]].copy()

result = result.sort_values(["체크", "소재지"], ascending=[False, True])

# 저장
result.to_excel("정부24_체크대상.xlsx", index=False)

print("완료")
print("체크 대상 개수:", result["체크"].sum())
display(result.head(30))

완료
체크 대상 개수: 36


,체크,소재지,면적,토지이동사유,고시면적,토지면적,출자면적,내부패턴,내부대표면적
13,1,울산광역시 남구 매암동 1-10,1582.0,필지 분할,884.0,884.0,884.0,고시=토지=출자,884.0
24,1,울산광역시 남구 매암동 139-29,46388.0,필지 분할,49080.0,NaN,NaN,고시만,49080.0
25,1,울산광역시 남구 매암동 139-30,1340.0,필지 분할,1372.0,NaN,NaN,고시만,1372.0
48,1,울산광역시 남구 매암동 139-82,1551.0,필지 분할,2519.0,NaN,NaN,고시만,2519.0
89,1,울산광역시 남구 매암동 472-42,2086.0,면적정정,1800.0,1800.0,1800.0,고시=토지=출자,1800.0
104,1,울산광역시 남구 여천동 187-10,50780.0,면적정정,49427.0,49427.0,49427.0,고시=토지=출자,49427.0
105,1,울산광역시 남구 여천동 187-12,114001.0,면적정정,107564.0,107564.0,107564.0,고시=토지=출자,107564.0
160,1,울산광역시 남구 장생포동 144-35,5.0,필지 분할,131.0,NaN,NaN,고시만,131.0
161,1,울산광역시 남구 장생포동 144-36,635.0,필지 분할,660.0,660.0,660.0,고시=토지=출자,660.0
162,1,울산광역시 남구 장생포동 144-37,24.0,필지 분할,26.0,26.0,26.0,고시=토지=출자,26.0
